<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/cubernetic/%D0%BA%D0%B8%D0%B1%D0%B5%D1%80%D0%BD%D0%B5%D1%82%D0%B8%D0%BA%D0%B02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq
from collections import defaultdict

def calculate_symbol_probabilities(text):
    freq = defaultdict(int)
    for char in text:
        if char == ' ':
            continue
        freq[char] += 1

    total = sum(freq.values())
    if total == 0:
        print("Ошибка: после удаления пробелов нет символов для обработки.")
        return {}, {}

    probs = {char: count / total for char, count in freq.items()}
    return probs, freq


def shannon_fano_coding(symbol_probs):
    sorted_items = sorted(symbol_probs.items(), key=lambda x: x[1], reverse=True)

    def divide_and_code(items, prefix=""):
        if len(items) == 1:
            return {items[0][0]: prefix}
        if len(items) == 2:
            return {items[0][0]: prefix + "0", items[1][0]: prefix + "1"}

        total_prob = sum(item[1] for item in items)
        min_diff = float('inf')
        best_split = 1

        for i in range(1, len(items)):
            left_sum = sum(item[1] for item in items[:i])
            right_sum = total_prob - left_sum
            diff = abs(left_sum - right_sum)
            if diff < min_diff:
                min_diff = diff
                best_split = i

        left_part = items[:best_split]
        right_part = items[best_split:]

        codes = {}
        codes.update(divide_and_code(left_part, prefix + "0"))
        codes.update(divide_and_code(right_part, prefix + "1"))
        return codes

    return divide_and_code(sorted_items)


def huffman_coding(symbol_probs):
    heap = [[prob, [symbol, ""]] for symbol, prob in symbol_probs.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]
        merged = [lo[0] + hi[0]] + lo[1:] + hi[1:]
        heapq.heappush(heap, merged)

    huff_tree = heapq.heappop(heap)
    return {pair[0]: pair[1] for pair in huff_tree[1:]}

def encode_message(message, coding_map):
    encoded = ""
    for char in message:
        if char != ' ' and char in coding_map:  # Не кодируем пробелы
            encoded += coding_map[char]
    return encoded

def decode_message(encoded, coding_map):
    reverse_map = {code: char for char, code in coding_map.items()}

    decoded = ""
    current = ""
    for bit in encoded:
        current += bit
        if current in reverse_map:
            decoded += reverse_map[current]
            current = ""
    return decoded

def main():
    print("Введите фразу: ", end="")
    phrase = input().strip()

    if not phrase:
        print("Ошибка: фраза не может быть пустой.")
        return

    filtered_phrase = phrase.replace(' ', '')

    if not filtered_phrase:
        print("Ошибка: после удаления пробелов фраза пуста.")
        return

    symbol_probs, freq = calculate_symbol_probabilities(phrase)

    print(f"\n{'Символ':<6} | {'Вероятность':<10}")
    print("-" * 25)
    for char in sorted(symbol_probs.keys()):
        print(f"{char:<6} | {symbol_probs[char]:<10.3f}")

    print(f"\nОбщее количество уникальных символов (без пробела): {len(symbol_probs)}")

    shannon_codes = shannon_fano_coding(symbol_probs)
    encoded_shannon = encode_message(phrase, shannon_codes)

    print(f"\nКодирование методом Шеннона-Фано:")
    print(f"{'Символ':<6} | {'Вероятность':<10} | {'Код':<8}")
    print("-" * 40)
    for char in sorted(shannon_codes.keys()):
        print(f"{char:<6} | {symbol_probs[char]:<10.3f} | {shannon_codes[char]:<8}")

    print(f"\nЗакодированная фраза (Фано): {encoded_shannon}")

    huffman_codes = huffman_coding(symbol_probs)
    encoded_huffman = encode_message(phrase, huffman_codes)

    print(f"\nКодирование методом Хаффмана:")
    print(f"{'Символ':<6} | {'Вероятность':<10} | {'Код':<8}")
    print("-" * 40)
    for char in sorted(huffman_codes.keys()):
        print(f"{char:<6} | {symbol_probs[char]:<10.3f} | {huffman_codes[char]:<8}")

    print(f"\nЗакодированная фраза (Хаффман): {encoded_huffman}")

    decoded_shannon = decode_message(encoded_shannon, shannon_codes)
    decoded_huffman = decode_message(encoded_huffman, huffman_codes)

    print(f"\nДекодированная фраза (Фано): {decoded_shannon}")
    print(f"Декодированная фраза (Хаффман): {decoded_huffman}")


if __name__ == "__main__":
    main()

Введите фразу: привет дорогие друзья

Символ | Вероятность
-------------------------
в      | 0.053     
г      | 0.053     
д      | 0.105     
е      | 0.105     
з      | 0.053     
и      | 0.105     
о      | 0.105     
п      | 0.053     
р      | 0.158     
т      | 0.053     
у      | 0.053     
ь      | 0.053     
я      | 0.053     

Общее количество уникальных символов (без пробела): 13

Кодирование методом Шеннона-Фано:
Символ | Вероятность | Код     
----------------------------------------
в      | 0.053      | 10110   
г      | 0.053      | 1100    
д      | 0.105      | 011     
е      | 0.105      | 010     
з      | 0.053      | 1110    
и      | 0.105      | 001     
о      | 0.105      | 100     
п      | 0.053      | 1010    
р      | 0.158      | 000     
т      | 0.053      | 10111   
у      | 0.053      | 1101    
ь      | 0.053      | 11110   
я      | 0.053      | 11111   

Закодированная фраза (Фано): 1010000001101100101011101110000010011000010100110001101111